In [64]:
import sounddevice as sd
import numpy as np
from faster_whisper import WhisperModel
import torch

In [51]:
SAMPLING_RATE = 16000  # Whisper expects 16 kHz
DURATION = 5           # seconds per recording chunk
model = WhisperModel("small")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
D:\Assignments\NLS_VA\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prasa\.cache\huggingface\hub\models--Systran--faster-whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article

In [60]:
print("Recording...")
audio = sd.rec(int(DURATION * SAMPLING_RATE), samplerate=SAMPLING_RATE, channels=1, dtype='float32')
sd.wait()
print("Recording finished.")

Recording...
Recording finished.


In [61]:
print("Playing back...")
sd.play(audio, SAMPLING_RATE)
sd.wait()
print("Done.")

Playing back...
Done.


In [62]:
audio = audio.squeeze()
segments, info = model.transcribe(
    audio,
    language="en",
    beam_size=5
)
transcription = "".join([seg.text for seg in segments])
print("Transcription:", transcription)

Transcription:  Tell me what is the weather tomorrow in


In [67]:
from transformers import Wav2Vec2Processor, HubertForCTC

processor = Wav2Vec2Processor.from_pretrained("facebook/hubert-large-ls960-ft")
model = HubertForCTC.from_pretrained("facebook/hubert-large-ls960-ft")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

HubertForCTC(
  (hubert): HubertModel(
    (feature_extractor): HubertFeatureEncoder(
      (conv_layers): ModuleList(
        (0): HubertLayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x HubertLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x HubertLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): HubertFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in

In [68]:
waveform = torch.tensor(audio).to(device)

input_values = processor(waveform, sampling_rate=SAMPLING_RATE, return_tensors="pt", padding=True).input_values
input_values = input_values.to(device)
with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.decode(predicted_ids[0]).lower()

print("\n📝 HuBERT Transcription:")
print(transcription)


📝 HuBERT Transcription:
tell me what is the wetter to morrow in


In [2]:
!pip install gpt4all


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install langchain

  Using cached xxhash-3.6.0-cp312-cp312-win_amd64.whl.metadata (13 kB)
Using cached xxhash-3.6.0-cp312-cp312-win_amd64.whl (31 kB)

   ------ ---------------------------------  2/13 [ormsgpack]
   ------------ ---------------------------  4/13 [jsonpatch]
   --------------- ------------------------  5/13 [requests-toolbelt]
   --------------- ------------------------  5/13 [requests-toolbelt]
   --------------- ------------------------  5/13 [requests-toolbelt]
   --------------- ------------------------  5/13 [requests-toolbelt]
   ------------------ ---------------------  6/13 [langsmith]
   ------------------ ---------------------  6/13 [langsmith]
   ------------------ ---------------------  6/13 [langsmith]
   ------------------ ---------------------  6/13 [langsmith]
   ------------------ ---------------------  6/13 [langsmith]
   ------------------ ---------------------  6/13 [langsmith]
   ------------------ ---------------------  6/13 [langsmith]
   ------------------ --------


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install --upgrade langchain


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import langchain
print(langchain.__version__)

1.0.8


In [14]:
import gpt4all

In [5]:
dir(gpt4all)

['CancellationError',
 'Embed4All',
 'GPT4All',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_pyllmodel',
 'gpt4all']

In [15]:
from gpt4all import GPT4All

ModuleNotFoundError: No module named 'langchain.llms'

In [24]:
gpt=GPT4All(model_name="ggml-gpt4all-j-v1.3-groovy.bin", model_path="./source/models/",allow_download=False)

RuntimeError: Unable to instantiate model: Unsupported file format

In [9]:
!wget https://gpt4all.io/models/ggml-gpt4all-mini.bin

'wget' is not recognized as an internal or external command,
operable program or batch file.
